In [1]:
# %% [0] Setup — metric, Weibull formula, postprocess, numerically-safe helpers
import re, unicodedata
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageOps

tf.random.set_seed(0); np.random.seed(0)

INPUT_DIR = sorted(Path("/kaggle/input").glob("**/Training_labels_updated.csv"))[0].parent
OUTPUT_DIR = Path("/kaggle/working"); OUTPUT_DIR.mkdir(exist_ok=True)

SIZES = np.array([0.002, 0.0063, 0.02, 0.063, 0.2, 0.63, 2, 6.3, 20, 63, 200])
SIZE_COLS = ["0.002","0.0063","0.02","0.063","0.2","0.63","2","6.3","20","63","200"]
W_EMD = np.diff(np.log10(SIZES))                      # exact competition weights, left-endpoint rule
CAM_EDGE, CAM_A52 = "Motorola Edge", "Samsung A52"
TARGET_PPM, TRAIN_LIKE_PPM, CROP_PX = 4.6, 4.6, 400
DOG_SIGMAS_MM = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0]
FFT_BANDS_MM = [(0.5,0.7), (0.7,1.4), (1.4,2.8), (2.8,5.6), (5.6,11.2)]

def emd(y_true, y_pred):
    return (np.abs(np.asarray(y_true)[..., :10] - np.asarray(y_pred)[..., :10]) * W_EMD).sum(-1)

def postprocess(curve):
    c = np.maximum.accumulate(np.clip(np.asarray(curve, float), 0, 100), axis=-1)
    c[..., -1] = 100.0
    return c

def weibull_curve(lnb, c, d=SIZES):
    lnb = np.asarray(lnb, float)[..., None]; c = np.maximum(np.asarray(c, float), 0.02)[..., None]
    return 100.0 * (1.0 - np.exp(-np.exp(c * (np.log(d) - lnb))))

def stable_softplus(x):
    """log(1+exp(x)), overflow-safe for large x."""
    return np.maximum(x, 0) + np.log1p(np.exp(-np.abs(x)))

# %% [1] Index photos — robust recursive glob, physical scale from each file's own size
CAMERA_RE = re.compile(r"^(Motorola_Edge_60_fusion|Motorola_Edge|Samsung_A52|iPhone_?14|iPhone_?16)_(.+?)(?:_\d+| \(\d+\))?$")
CAMERA_KEY = {"Motorola_Edge_60_fusion": "Motorola Edge 60 Fusion", "Motorola_Edge": CAM_EDGE,
              "Samsung_A52": CAM_A52, "iPhone14": "iPhone 14", "iPhone_14": "iPhone 14",
              "iPhone16": "iPhone 16", "iPhone_16": "iPhone 16"}

def normalize_id(raw):
    s = unicodedata.normalize("NFC", raw).replace("ü","ue").replace("ö","oe").replace("ä","ae")
    return s.replace(",", "_")

labels = pd.read_csv(INPUT_DIR/"Training_labels_updated.csv", dtype={"sample_id": str}).set_index("sample_id")
sample_sub = pd.read_csv(INPUT_DIR/"sample_submission.csv", dtype={"sample_id": str})
ppm_tab = pd.read_csv(INPUT_DIR/"ppm_updated.csv").set_index("phone")

rows = []
for split, pattern in [("train", "Training-All_Photos*"), ("test", "Test_All_Photos*")]:
    for p in sorted(INPUT_DIR.glob(f"{pattern}/**/*")):
        if p.suffix.lower() not in {".jpg", ".jpeg"}: continue
        m = CAMERA_RE.match(unicodedata.normalize("NFC", p.stem))
        if m is None: raise ValueError(f"unexpected file name: {p.name}")
        with Image.open(p) as im: w, h = im.size
        rows.append({"split": split, "path": p, "camera": CAMERA_KEY[m.group(1)],
                     "sample_id": normalize_id(m.group(2)), "w": w, "h": h})
photos = pd.DataFrame(rows)
photos["ppm_eff"] = [ppm_tab.loc[c,"ppm"] * max(w,h) / ppm_tab.loc[c,"width"]
                     for c,w,h in zip(photos.camera, photos.w, photos.h)]
assert sorted(set(photos.query("split=='train'").sample_id)) == sorted(labels.index)
assert set(photos.query("split=='test'").sample_id) == set(sample_sub.sample_id)
tr, te = photos[photos.split=="train"].reset_index(drop=True), photos[photos.split=="test"].reset_index(drop=True)
print(photos.groupby(["split","camera"]).agg(photos=("path","size"), px_per_mm=("ppm_eff","median")).round(1))

# %% [2] Shared preprocessing — ONE function returns the CNN crop, DoG features, and FFT features
_HANN_CACHE = {}

def load_all(path, ppm_eff):
    im = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    a = np.asarray(im)
    if ppm_eff > 6.0:                                   # sharp (test) photo -> degrade toward train condition
        f = TRAIN_LIKE_PPM / ppm_eff
        a = cv2.resize(a, (round(a.shape[1]*f), round(a.shape[0]*f)), interpolation=cv2.INTER_AREA)
        ppm_eff = TRAIN_LIKE_PPM
    f = TARGET_PPM / ppm_eff
    a = cv2.resize(a, (round(a.shape[1]*f), round(a.shape[0]*f)), interpolation=cv2.INTER_AREA)
    if a.shape[0] > a.shape[1]: a = np.rot90(a)
    H, W = a.shape[:2]

    # (a) fixed-size centre crop, RGB, for EfficientNet
    s = min(CROP_PX, H, W); y, x = (H-s)//2, (W-s)//2
    crop_cnn = cv2.resize(a[y:y+s, x:x+s], (CROP_PX, CROP_PX), interpolation=cv2.INTER_AREA).astype(np.float32)

    # (b) 50% centre crop, grayscale, for DoG + FFT
    c = int(0.25*min(H,W))
    gray = cv2.cvtColor(a[c:H-c, c:W-c], cv2.COLOR_RGB2GRAY).astype(np.float32)

    blurs = [gray] + [cv2.GaussianBlur(gray, (0,0), sig*TARGET_PPM) for sig in DOG_SIGMAS_MM]
    dog = np.array([np.abs(blurs[i]-blurs[i+1]).mean() for i in range(len(DOG_SIGMAS_MM))], dtype=np.float32)

    n = min(gray.shape); g = gray[:n, :n]
    if n not in _HANN_CACHE: _HANN_CACHE[n] = np.outer(np.hanning(n), np.hanning(n)).astype(np.float32)
    gc = (g - g.mean()) / (g.std() + 1e-6)
    P = np.abs(np.fft.fftshift(np.fft.fft2(gc * _HANN_CACHE[n]))) ** 2
    yx = np.indices((n, n)) - n // 2
    freq_cycles_per_px = np.sqrt(yx[0]**2 + yx[1]**2) / n
    wavelength_mm = np.divide(1.0, freq_cycles_per_px * TARGET_PPM, out=np.full((n,n), np.inf), where=freq_cycles_per_px>0)
    fft_feats = []
    for lo, hi in FFT_BANDS_MM:
        mask = (wavelength_mm >= lo) & (wavelength_mm < hi)
        fft_feats.append(np.log(P[mask].mean() + 1e-6) if mask.any() else 0.0)
    fft = np.array(fft_feats, dtype=np.float32)

    return crop_cnn, np.concatenate([dog, fft])

# %% [3] Precompute DoG+FFT features once per photo (cheap, CPU) and fit a scaler for them
print("computing DoG + FFT features for every photo...")
hand_cache = {}
for df in (tr, te):
    for r in df.itertuples():
        _, hand = load_all(r.path, r.ppm_eff)
        hand_cache[str(r.path)] = hand

N_HAND = len(DOG_SIGMAS_MM) + len(FFT_BANDS_MM)
hand_tr_mat = np.stack([hand_cache[str(p)] for p in tr.path])
HAND_MEAN, HAND_STD = hand_tr_mat.mean(0), hand_tr_mat.std(0) + 1e-6   # fit scaler on TRAIN only

def hand_scaled(path):
    return (hand_cache[str(path)] - HAND_MEAN) / HAND_STD

# %% [4] tf.data pipeline — yields (image, hand_features) as TWO inputs, y as the target
def make_dataset(df, training):
    paths = df.path.values; ppms = df.ppm_eff.values
    has_labels = all(sid in labels.index for sid in df.sample_id)          # True for tr, False for te
    ys = (labels.loc[df.sample_id, SIZE_COLS[:10]].values.astype(np.float32)
          if has_labels else np.zeros((len(df), 10), dtype=np.float32))    # placeholder for test; never trained on

    def gen():
        for p, ppm, y in zip(paths, ppms, ys):
            crop_cnn, _ = load_all(p, ppm)              # recompute the CNN crop fresh each epoch (cheap)
            hand = hand_scaled(p)
            yield (crop_cnn, hand), y

    ds = tf.data.Dataset.from_generator(gen, output_signature=(
        (tf.TensorSpec((CROP_PX,CROP_PX,3), tf.float32), tf.TensorSpec((N_HAND,), tf.float32)),
        tf.TensorSpec((10,), tf.float32)))

    if training:
        def augment(inputs, y):
            img, hand = inputs
            img = tf.image.random_flip_left_right(img); img = tf.image.random_flip_up_down(img)
            img = tf.image.rot90(img, tf.random.uniform([], 0, 4, dtype=tf.int32))
            img = tf.image.random_brightness(img, 0.15); img = tf.image.random_contrast(img, 0.85, 1.15)
            return (img, hand), y
        ds = ds.repeat().map(augment, num_parallel_calls=tf.data.AUTOTUNE)   # infinite stream + steps_per_epoch
    return ds.batch(16).prefetch(tf.data.AUTOTUNE)

# %% [5] The fused model — EfficientNet (frozen) + DoG+FFT, concatenated, then a 4-hidden-layer head
eff = tf.keras.applications.EfficientNetV2S(include_top=False, weights="imagenet", pooling="avg")
eff.trainable = False                                      # frozen, exactly as in GRAI3

img_in = tf.keras.Input((CROP_PX, CROP_PX, 3), name="image")
hand_in = tf.keras.Input((N_HAND,), name="hand_features")

x_img = tf.keras.applications.efficientnet_v2.preprocess_input(img_in)
x_img = eff(x_img, training=False)                          # -> pooled feature vector, frozen

fused = tf.keras.layers.Concatenate()([x_img, hand_in])     # late fusion: learned + hand-engineered features

h = tf.keras.layers.Dense(2048, activation="relu")(fused)    # hidden layer 1
h = tf.keras.layers.Dropout(0.1)(h)
h = tf.keras.layers.Dense(512, activation="relu")(h)          # hidden layer 2
h = tf.keras.layers.Dropout(0.1)(h)
h = tf.keras.layers.Dense(256, activation="relu")(h)          # hidden layer 3
h = tf.keras.layers.Dropout(0.1)(h)
h = tf.keras.layers.Dense(128, activation="relu")(h)          # hidden layer 4
h = tf.keras.layers.Dropout(0.1)(h)
out = tf.keras.layers.Dense(2, activation="linear", name="weibull_params")(h)   # output: raw (ln_b, c_raw)

model = tf.keras.Model([img_in, hand_in], out)
model.summary()

# %% [6] Train directly on the EXACT competition metric (no intermediate Weibull-label-fitting step)
D_SUPPORT = tf.constant(SIZES[:10], dtype=tf.float32)
W_TF = tf.constant(W_EMD, dtype=tf.float32)

def emd_loss(y_true, y_pred):
    ln_b, c = y_pred[:,0], tf.nn.softplus(y_pred[:,1]) + 0.02   # softplus keeps c > 0, as Weibull requires
    ratio = D_SUPPORT[None,:] / tf.exp(ln_b)[:,None]
    F = 100.0 * (1.0 - tf.exp(-tf.pow(ratio, c[:,None])))
    return tf.reduce_sum(tf.abs(F - y_true) * W_TF[None,:], axis=1)

steps = max(1, len(tr) // 16)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=emd_loss)
model.fit(make_dataset(tr, training=True), epochs=25, steps_per_epoch=steps,
          callbacks=[tf.keras.callbacks.EarlyStopping(monitor="loss", patience=5, restore_best_weights=True)])

# %% [7] Predict, aggregate per sample (mean of per-photo curves), postprocess
def predict_curves(df):
    ds = make_dataset(df, training=False)
    pred = model.predict(ds, verbose=0)
    ln_b, c = pred[:,0], stable_softplus(pred[:,1]) + 0.02       # overflow-safe softplus
    curves = pd.DataFrame(weibull_curve(ln_b, c), index=df.sample_id, columns=SIZE_COLS)
    return postprocess(curves.groupby(level=0).mean().to_numpy()), curves.index.unique()

pred_tr, ids_tr = predict_curves(tr)
print("Fused (EfficientNet+DoG+FFT) model EMD on TRAIN (optimistic — not held-out):",
      emd(labels.loc[ids_tr, SIZE_COLS].values, pred_tr).mean().round(2))

pred_te, ids_te = predict_curves(te)
pred_te_df = pd.DataFrame(pred_te, index=ids_te, columns=SIZE_COLS)

# %% [8] SAM — measure cobbles directly, redraw the 20/63 mm points where stones are visible
import torch
from transformers import pipeline
mask_gen = pipeline("mask-generation", model="facebook/sam-vit-base", device=0 if torch.cuda.is_available() else -1)
WORK_PPM, WORK_CROP = 2.0, 0.12

def load_work(path, ppm):
    im = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    a = cv2.resize(np.asarray(im), None, fx=WORK_PPM/ppm, fy=WORK_PPM/ppm, interpolation=cv2.INTER_AREA)
    if a.shape[0] > a.shape[1]: a = np.rot90(a)
    H, W = a.shape[:2]; c = WORK_CROP
    return np.ascontiguousarray(a[int(H*c):int(H*(1-c)), int(W*c):int(W*(1-c))])

def stones_from_masks(masks, tot):
    if not masks:                                       # SAM found nothing -- no crash, no stones
        return []
    order = sorted(range(len(masks)), key=lambda i: -int(masks[i].sum())); taken = np.zeros_like(masks[0], bool)
    out = []
    for i in order:
        m = masks[i]; sz = int(m.sum())
        if sz < (8.0*WORK_PPM)**2 or sz > 0.5*tot or (m & taken).sum() > 0.5*sz: continue
        ys, xs = np.nonzero(m); major = max(ys.max()-ys.min(), xs.max()-xs.min()) / WORK_PPM
        fill = sz / (np.pi*(major*WORK_PPM/2)**2 + 1e-6)
        if fill < 0.35: continue
        taken |= m; out.append({"area_frac": sz/tot, "major_mm": major})
    return out

targets = photos[photos.ppm_eff >= 12.0]              # full-res only: test photos + H374 calibration sample
meas = []
for r in targets.itertuples():
    rgb = load_work(r.path, r.ppm_eff)
    out = mask_gen(Image.fromarray(rgb), points_per_batch=64, pred_iou_thresh=0.86, stability_score_thresh=0.9)
    stones = stones_from_masks([np.asarray(m) for m in out["masks"]], rgb.shape[0]*rgb.shape[1])
    meas.append({"sample_id": r.sample_id,
                "A63": sum(s["area_frac"] for s in stones if s["major_mm"] > 63),
                "A20": sum(s["area_frac"] for s in stones if s["major_mm"] > 20)})
ms = pd.DataFrame(meas).groupby("sample_id").mean()

I20, I63 = SIZE_COLS.index("20"), SIZE_COLS.index("63")
if "H374" in ms.index and ms.loc["H374","A20"] > 0:
    K_CAL = (100.0 - labels.loc["H374","20"]) / (100.0 * ms.loc["H374","A20"])
else:
    K_CAL = 1.0   # Delesse fallback: area fraction ~ mass fraction, if the calibration sample wasn't measured
print(f"k = {K_CAL:.3f}")

def redraw(curve, a63, a20):
    c = curve.copy()
    new20 = max(100.0 - K_CAL*100.0*a20, 0.0); new63 = max(100.0 - K_CAL*100.0*a63, new20)
    scale = new20 / max(c[I20], 1e-6)
    if scale < 1: c[:I20+1] *= scale
    c[I20], c[I63], c[-1] = new20, new63, 100.0
    return c

final = {}
for sid in pred_te_df.index:
    a63, a20 = (float(ms.loc[sid,"A63"]), float(ms.loc[sid,"A20"])) if sid in ms.index else (0.0, 0.0)
    final[sid] = redraw(pred_te_df.loc[sid].to_numpy(), a63, a20) if a63 >= 0.02 else pred_te_df.loc[sid].to_numpy()
pred_final = pd.DataFrame(postprocess(np.stack([final[s] for s in pred_te_df.index])), index=pred_te_df.index, columns=SIZE_COLS)

# %% [9] Submission + structural checks
sub = sample_sub[["sample_id"]].merge(pred_final, left_on="sample_id", right_index=True, how="left")
vals = sub[SIZE_COLS].to_numpy()
assert not np.isnan(vals).any()
assert ((vals>=0)&(vals<=100)).all() and (np.diff(vals,axis=1)>=0).all() and (vals[:,-1]==100).all()
sub.to_csv(OUTPUT_DIR/"submission.csv", index=False, float_format="%.4f")
print(sub.round(1).to_string(index=False))

                               photos  px_per_mm
split camera                                    
test  iPhone 14                    14       13.9
      iPhone 16                    21       19.5
train Motorola Edge                69        4.6
      Motorola Edge 60 Fusion       3       12.5
      Samsung A52                  55        4.6
computing DoG + FFT features for every photo...


2026-10-31 09:40:38.185056: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 400, 400,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficientnetv2-s    │ (None, 1280)      │ 20,331,360 │ image[0][0]       │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hand_features       │ (None, 11)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1291)      │          0 │ efficientnetv2-s… │
│ (Concatenate)       │                   │            │ hand_features[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 2048)      │  2,646,016 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 2048)      │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 512)       │  1,049,088 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 512)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 256)       │    131,328 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 256)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │     32,896 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128)       │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ weibull_params      │ (None, 2)         │        258 │ dropout_3[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,190,946 (92.28 MB)

... (8 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

k = 1.305
               sample_id  0.002  0.0063  0.02  0.063  0.2  0.63    2   6.3    20    63   200
        HPC_Airbus BS6-3    1.2     3.0   7.4   17.8 39.2  70.9 94.7  99.8 100.0 100.0 100.0
          HPC_Audorfring    1.7     4.1   9.9   22.3 45.5  75.5 95.2  99.8 100.0 100.0 100.0
HPC_Muenster_BS6_9_0-10m    0.1     0.2   0.5    1.1  2.4   4.1  5.3   5.5   5.5  65.2 100.0
   HPC_Airbus BS10-4bis7    0.7     1.7   4.1    9.5 21.2  42.9 72.4  93.9  99.7 100.0 100.0
   HPC_Airbus BS12-5bis8    1.8     4.2   9.9   22.2 45.1  74.9 95.0  99.8 100.0 100.0 100.0
 HPC_Kleinkummerfeld 2-2    1.2     2.8   6.6   15.2 32.6  59.7 86.3  98.5 100.0 100.0 100.0
 HPC_Kleinkummerfeld 2-3    0.3     0.7   1.7    3.8  8.6  18.5 37.5  65.8  91.3  99.6 100.0
 HPC_Kleinkummerfeld 9-4    0.4     0.9   2.0    4.5  9.9  21.2 42.0  71.1  94.1  99.8 100.0
HPC_Kleinkummerfeld 18-3    1.7     4.5  11.4   27.1 55.8  86.9 99.2 100.0 100.0 100.0 100.0
   HPC_Testfeld Lidl WHV    1.3     3.1   7.4   17.1 36.5  6